# Week 11: Graphs — Representation, BFS & DFS
## PHASE 5: Network Structures

*📚 Data Structures & Algorithms · ⏱️ 3 Hours · 👨‍🏫 Dr. Arif Solmaz*

## 🎯 Learning Objectives

1. Understand what a **graph** is: nodes (vertices) and edges
2. Identify real-world examples of graphs
3. Distinguish between **directed** and **undirected** graphs
4. Represent graphs using an **adjacency list** (Python dict)
5. Implement **BFS** (Breadth-First Search) using a queue
6. Implement **DFS** (Depth-First Search) using a stack / recursion
7. Find the **shortest path** (by number of steps) using BFS
8. Compare BFS and DFS behavior

## 🎯 Core Mastery Connection

When data forms a network (not a hierarchy), graphs are the right structure. BFS finds shortest paths by number of edges; DFS explores exhaustively. Choosing between them depends on the problem — and you can benchmark both to confirm they have the same O(V + E) complexity but different practical behaviors.

---
## 🧭 Three-Hour Interactive Studio Plan

**Audience:** Mechatronics Engineering students  
**Weekly focus:** Week 11: Graphs — Representation, BFS & DFS

**Professional lens:** representing factory networks, robot maps, and dependency systems.

| Time | Learning cycle |
|---|---|
| 00:00–00:10 | Launch question, prior-knowledge retrieval, outcomes |
| 00:10–00:50 | Concept cycle 1: explain → predict → test |
| 00:50–01:00 | Checkpoint 1, student questions, peer explanation |
| 01:00–01:10 | Break |
| 01:10–01:50 | Concept cycle 2: worked example → variation → discussion |
| 01:50–02:00 | Checkpoint 2 and misconception repair |
| 02:00–02:10 | Break |
| 02:10–02:40 | Core in-class practice with instructor circulation |
| 02:40–02:50 | Checkpoint 3: exam bridge and professional transfer |
| 02:50–03:00 | Open questions, summary, and exit ticket |

The official start and finish times are followed as published in the timetable. Ask questions at any point; the scheduled checkpoints guarantee additional question time. Checkpoints are private self-checks in this runtime—no identity, upload, homework, or instructor dashboard.


In [ ]:
# Run once. This pulse stays only in the current Colab runtime.
_studio_pulses = {}

def studio_pulse(number, response, minimum_words=8):
    words = str(response).strip().split()
    ready = len(words) >= minimum_words
    _studio_pulses[int(number)] = ready
    if ready:
        print(f"✅ Checkpoint {number}: explanation recorded locally ({len(words)} words).")
    else:
        print(f"🟡 Checkpoint {number}: explain your reasoning in at least {minimum_words} words, then retry.")
    print("Nothing is transmitted or stored for grading.")
    return ready

print("✅ Local studio checkpoints ready")

---
## 📦 Setup

Run this cell first to load the required packages.

In [ ]:
import matplotlib.pyplot as plt

import collections
import random
import time

---
## Part 1: What Is a Graph?

A **graph** is a collection of **nodes** (also called **vertices**) connected by **edges**.

Unlike trees, graphs have no "root" and can have **cycles** (paths that loop back).

```
    A --- B
    |     |
    C --- D --- E
```

| Term | Meaning |
|------|--------|
| **Node / Vertex** | A point in the graph (A, B, C, ...) |
| **Edge** | A connection between two nodes |
| **Neighbor** | A node directly connected by an edge |
| **Path** | A sequence of nodes connected by edges |
| **Cycle** | A path that starts and ends at the same node |
| **Degree** | Number of edges connected to a node |

**Real-world examples:**

| Graph | Nodes | Edges |
|-------|-------|-------|
| Social network | People | Friendships |
| Map / Roads | Cities | Roads between cities |
| Web pages | Pages | Hyperlinks |
| Course prerequisites | Courses | "Must take before" |

**Figure 1.1** — A simple graph as a Python dictionary

In [ ]:
# A simple friendship graph:
#   Alice --- Bob
#   |         |
#   Carol --- Dave --- Eve

friends = {
    "Alice": ["Bob", "Carol"],
    "Bob":   ["Alice", "Dave"],
    "Carol": ["Alice", "Dave"],
    "Dave":  ["Bob", "Carol", "Eve"],
    "Eve":   ["Dave"]
}

# Who are Alice's friends?
print(f"Alice's friends: {friends['Alice']}")

# How many friends does Dave have?
print(f"Dave's friends: {friends['Dave']} (degree: {len(friends['Dave'])})")

# How many people in total?
print(f"Total people: {len(friends)}")

---
## Part 2: Directed vs Undirected Graphs

| Type | Edges | Example |
|------|-------|---------|
| **Undirected** | Go both ways (A—B means B—A too) | Friendships, roads |
| **Directed** | One-way (A→B doesn’t mean B→A) | Twitter follows, web links |

```
Undirected:          Directed:
  A --- B              A --> B
  |     |              |     |
  C --- D              C <-- D
                       ↑
                       A --> C too
```

**Real-world analogy:**
- **Undirected:** A two-way street — you can drive in both directions
- **Directed:** A one-way street — you can only go in one direction

**Figure 2.1** — Directed graph (Twitter-like follows)

In [ ]:
# Directed graph: A follows B doesn't mean B follows A
follows = {
    "Alice": ["Bob", "Carol"],     # Alice follows Bob and Carol
    "Bob":   ["Alice"],            # Bob follows Alice
    "Carol": ["Dave"],             # Carol follows Dave
    "Dave":  ["Alice", "Bob"],     # Dave follows Alice and Bob
    "Eve":   ["Alice", "Dave"],    # Eve follows Alice and Dave
}

# Alice follows Bob, but does Bob follow Alice?
print(f"Alice follows: {follows['Alice']}")
print(f"Bob follows:   {follows['Bob']}")
print(f"Does Alice follow Bob? {'Bob' in follows['Alice']}")
print(f"Does Bob follow Alice? {'Alice' in follows['Bob']}")
print()

# Eve follows Alice, but Alice doesn't follow Eve
print(f"Eve follows: {follows['Eve']}")
print(f"Does Alice follow Eve? {'Eve' in follows['Alice']}")

**Figure 2.2** — Building an undirected graph with a helper function

In [ ]:
def add_edge(graph, node1, node2):
    """Add an undirected edge between node1 and node2."""
    if node1 not in graph:
        graph[node1] = []
    if node2 not in graph:
        graph[node2] = []
    
    # Add in both directions (undirected)
    graph[node1].append(node2)
    graph[node2].append(node1)


# Build a graph of cities connected by roads
roads = {}
add_edge(roads, "Istanbul", "Ankara")
add_edge(roads, "Istanbul", "Bursa")
add_edge(roads, "Ankara", "Konya")
add_edge(roads, "Ankara", "Antalya")
add_edge(roads, "Bursa", "Eskisehir")
add_edge(roads, "Eskisehir", "Ankara")

print("Road connections:")
for city, neighbors in roads.items():
    print(f"  {city}: {neighbors}")

---
## Part 3: Adjacency List Representation

There are two common ways to represent a graph:

| Method | Description | Best for |
|--------|-----------|----------|
| **Adjacency List** | Dict of lists | Sparse graphs (few edges) |
| **Adjacency Matrix** | 2D grid of 0/1 | Dense graphs (many edges) |

We’ll use **adjacency lists** (Python dicts) because they’re simpler and more memory-efficient for most cases.

```
Graph:         Adjacency List:
  A --- B      A: [B, C]
  |     |      B: [A, D]
  C --- D      C: [A, D]
               D: [B, C]
```

**Figure 3.1** — A Graph class using adjacency list

In [ ]:
class Graph:
    """A simple undirected graph using adjacency list."""
    
    def __init__(self):
        self.adj_list = {}  # Dictionary: node → list of neighbors
    
    def add_node(self, node):
        """Add a node with no edges."""
        if node not in self.adj_list:
            self.adj_list[node] = []
    
    def add_edge(self, node1, node2):
        """Add an undirected edge."""
        self.add_node(node1)
        self.add_node(node2)
        self.adj_list[node1].append(node2)
        self.adj_list[node2].append(node1)
    
    def get_neighbors(self, node):
        """Return the neighbors of a node."""
        return self.adj_list.get(node, [])
    
    def display(self):
        """Print the adjacency list."""
        for node, neighbors in self.adj_list.items():
            print(f"  {node} → {neighbors}")


# Build a graph
g = Graph()
g.add_edge("A", "B")
g.add_edge("A", "C")
g.add_edge("B", "D")
g.add_edge("C", "D")
g.add_edge("D", "E")

print("Adjacency List:")
g.display()
print(f"\nNeighbors of D: {g.get_neighbors('D')}")

---
### ⏱️ Checkpoint 1 of 3 — Think · Pair · Explain

For **Week 11: Graphs — Representation, BFS & DFS**, state the key invariant, operation cost, or decision rule in your own words.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_1_response = ""  # write at least 8 words
studio_pulse(1, checkpoint_1_response)

---
## Part 4: BFS — Breadth-First Search

BFS explores a graph **level by level**, visiting all neighbors before going deeper.

**How it works:**
1. Start at a node, add it to a **queue**
2. While the queue is not empty:
   - Remove the **front** node from the queue
   - Visit it (if not already visited)
   - Add all its **unvisited neighbors** to the queue

**Real-world analogy:** Imagine you’re looking for your keys in a building:
- BFS = Check **all rooms on the current floor** before going to the next floor
- You explore outward like ripples in a pond

```
       A
      / \
     B   C       BFS order from A: A, B, C, D, E
     |   |       (level by level)
     D   E
```

**Figure 4.1** — BFS implementation with step-by-step tracing

In [ ]:
from collections import deque

def bfs(graph, start):
    """Breadth-First Search: explore level by level."""
    visited = set()       # Track which nodes we've visited
    queue = deque([start]) # Queue of nodes to explore
    order = []            # Order in which we visit nodes
    
    visited.add(start)
    
    while queue:
        # Remove from front of queue
        current = queue.popleft()
        order.append(current)
        print(f"  Visit: {current}  |  Queue: {list(queue)}")
        
        # Add unvisited neighbors to back of queue
        for neighbor in graph[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    
    return order


# Graph:
#   A --- B --- D
#   |         |
#   C --- E --- F

graph = {
    "A": ["B", "C"],
    "B": ["A", "D"],
    "C": ["A", "E"],
    "D": ["B", "F"],
    "E": ["C", "F"],
    "F": ["D", "E"]
}

print("BFS from A:")
result = bfs(graph, "A")
print(f"\nBFS order: {result}")

**Figure 4.2** — BFS level-by-level visualization

In [ ]:
from collections import deque

def bfs_levels(graph, start):
    """BFS that shows which level each node is on."""
    visited = set([start])
    queue = deque([(start, 0)])  # (node, level)
    levels = {}  # level → list of nodes
    
    while queue:
        current, level = queue.popleft()
        
        if level not in levels:
            levels[level] = []
        levels[level].append(current)
        
        for neighbor in graph[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, level + 1))
    
    return levels


levels = bfs_levels(graph, "A")
print("BFS Levels from A:")
for level, nodes in sorted(levels.items()):
    print(f"  Level {level}: {nodes}")

---
## Part 5: DFS — Depth-First Search

DFS explores a graph by going **as deep as possible** before backtracking.

**How it works:**
1. Start at a node, add it to a **stack** (or use recursion)
2. While the stack is not empty:
   - Pop the **top** node from the stack
   - Visit it (if not already visited)
   - Push all its **unvisited neighbors** onto the stack

**Real-world analogy:** Imagine exploring a maze:
- DFS = Go down one path as far as you can, then **backtrack** when you hit a dead end
- Like following one thread of a story to its conclusion before trying another

```
       A
      / \
     B   C       DFS order from A: A, B, D, C, E
     |   |       (go deep first)
     D   E
```

**Figure 5.1** — DFS implementation using a stack

In [ ]:
def dfs(graph, start):
    """Depth-First Search using a stack."""
    visited = set()
    stack = [start]   # Use a list as a stack
    order = []
    
    while stack:
        # Pop from top of stack
        current = stack.pop()
        
        if current not in visited:
            visited.add(current)
            order.append(current)
            print(f"  Visit: {current}  |  Stack: {stack}")
            
            # Push unvisited neighbors onto stack
            for neighbor in graph[current]:
                if neighbor not in visited:
                    stack.append(neighbor)
    
    return order


print("DFS from A:")
result = dfs(graph, "A")
print(f"\nDFS order: {result}")

**Figure 5.2** — DFS implementation using recursion

In [ ]:
def dfs_recursive(graph, node, visited=None, order=None):
    """DFS using recursion (the call stack IS the stack)."""
    if visited is None:
        visited = set()
        order = []
    
    visited.add(node)
    order.append(node)
    print(f"  Visit: {node}")
    
    for neighbor in graph[node]:
        if neighbor not in visited:
            dfs_recursive(graph, neighbor, visited, order)
    
    return order


print("DFS (recursive) from A:")
result = dfs_recursive(graph, "A")
print(f"\nDFS order: {result}")

---
## Part 6: BFS vs DFS Comparison

| Feature | BFS | DFS |
|---------|-----|-----|
| Data structure | Queue (FIFO) | Stack (LIFO) |
| Exploration | Level by level | Deep then backtrack |
| Shortest path? | Yes (unweighted) | No |
| Memory | More (stores all neighbors) | Less (just current path) |
| Best for | Shortest path, nearby nodes | Exploring all paths, cycles |

**Figure 6.1** — Side-by-side BFS vs DFS comparison

In [ ]:
from collections import deque

# A larger graph to see the difference clearly
#       1
#      / \
#     2   3
#    / \   \
#   4   5   6
#       |   |
#       7   8

graph2 = {
    1: [2, 3],
    2: [1, 4, 5],
    3: [1, 6],
    4: [2],
    5: [2, 7],
    6: [3, 8],
    7: [5],
    8: [6]
}

# BFS (quiet version)
def bfs_quiet(graph, start):
    visited = set([start])
    queue = deque([start])
    order = []
    while queue:
        current = queue.popleft()
        order.append(current)
        for neighbor in graph[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return order

# DFS (quiet version)
def dfs_quiet(graph, start):
    visited = set()
    stack = [start]
    order = []
    while stack:
        current = stack.pop()
        if current not in visited:
            visited.add(current)
            order.append(current)
            for neighbor in graph[current]:
                if neighbor not in visited:
                    stack.append(neighbor)
    return order

print(f"BFS from 1: {bfs_quiet(graph2, 1)}")
print(f"DFS from 1: {dfs_quiet(graph2, 1)}")
print()
print("Notice: BFS visits level by level (1, 2, 3, 4, 5, 6, ...)")
print("        DFS goes deep first (1, 3, 6, 8, 2, 5, 7, 4)")

---
## Part 7: Finding Shortest Path with BFS

BFS naturally finds the **shortest path** (fewest edges) between two nodes in an unweighted graph.

**How?** We keep track of **who discovered each node** (the parent), then trace back from the destination to the start.

**Real-world analogy:** You’re at a train station and want to get to another station with the fewest transfers. BFS checks all stations reachable in 1 transfer, then 2 transfers, etc.

**Figure 7.1** — BFS shortest path implementation

In [ ]:
from collections import deque

def bfs_shortest_path(graph, start, end):
    """Find the shortest path from start to end using BFS."""
    if start == end:
        return [start]
    
    visited = set([start])
    queue = deque([start])
    parent = {start: None}  # Track who discovered each node
    
    while queue:
        current = queue.popleft()
        
        for neighbor in graph[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                parent[neighbor] = current
                queue.append(neighbor)
                
                # Found the destination!
                if neighbor == end:
                    # Trace back the path
                    path = []
                    node = end
                    while node is not None:
                        path.append(node)
                        node = parent[node]
                    return path[::-1]  # Reverse to get start → end
    
    return None  # No path found


# City map:
#   Istanbul --- Bursa --- Eskisehir
#      |                     |
#   Ankara ---- Konya ---- Antalya

city_map = {
    "Istanbul":  ["Bursa", "Ankara"],
    "Bursa":     ["Istanbul", "Eskisehir"],
    "Eskisehir": ["Bursa", "Antalya"],
    "Ankara":    ["Istanbul", "Konya"],
    "Konya":     ["Ankara", "Antalya"],
    "Antalya":   ["Eskisehir", "Konya"]
}

path = bfs_shortest_path(city_map, "Istanbul", "Antalya")
print(f"Shortest path Istanbul → Antalya: {' → '.join(path)}")
print(f"Number of stops: {len(path) - 1}")

print()
path2 = bfs_shortest_path(city_map, "Bursa", "Konya")
print(f"Shortest path Bursa → Konya: {' → '.join(path2)}")
print(f"Number of stops: {len(path2) - 1}")

**Figure 7.2** — Visualizing the shortest path search

In [ ]:
from collections import deque

def bfs_shortest_path_verbose(graph, start, end):
    """BFS shortest path with detailed output."""
    visited = set([start])
    queue = deque([(start, [start])])  # (node, path_so_far)
    
    step = 0
    while queue:
        current, path = queue.popleft()
        step += 1
        print(f"  Step {step}: At {current}, path so far: {path}")
        
        if current == end:
            print(f"  ✅ Found destination!")
            return path
        
        for neighbor in graph[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor]))
    
    return None


print("Finding shortest path from Istanbul to Antalya:")
path = bfs_shortest_path_verbose(city_map, "Istanbul", "Antalya")
print(f"\nResult: {' → '.join(path)} ({len(path) - 1} stops)")

---
## Part 8: Studio Exercise — Social Network

Let’s put it all together with a practical example: a social network.

**Figure 8.1** — Finding degrees of separation in a social network

In [ ]:
from collections import deque

# Social network
social = {
    "Ahmet":    ["Mehmet", "Ayse"],
    "Mehmet":   ["Ahmet", "Fatma", "Ali"],
    "Ayse":     ["Ahmet", "Zeynep"],
    "Fatma":    ["Mehmet", "Zeynep"],
    "Ali":      ["Mehmet", "Hasan"],
    "Zeynep":   ["Ayse", "Fatma", "Hasan"],
    "Hasan":    ["Ali", "Zeynep"]
}

def degrees_of_separation(graph, person1, person2):
    """Find the degrees of separation between two people."""
    path = bfs_shortest_path(graph, person1, person2)
    if path:
        return len(path) - 1
    return -1

def friend_suggestions(graph, person):
    """Suggest friends-of-friends (people 2 hops away)."""
    direct_friends = set(graph[person])
    suggestions = set()
    
    for friend in graph[person]:
        for fof in graph[friend]:
            if fof != person and fof not in direct_friends:
                suggestions.add(fof)
    
    return list(suggestions)


# Degrees of separation
print("Degrees of separation:")
print(f"  Ahmet ↔ Hasan: {degrees_of_separation(social, 'Ahmet', 'Hasan')}")
print(f"  Ahmet ↔ Fatma: {degrees_of_separation(social, 'Ahmet', 'Fatma')}")
print(f"  Ahmet ↔ Mehmet: {degrees_of_separation(social, 'Ahmet', 'Mehmet')}")

print()

# Friend suggestions
print("Friend suggestions:")
print(f"  For Ahmet: {friend_suggestions(social, 'Ahmet')}")
print(f"  For Ali:   {friend_suggestions(social, 'Ali')}")

---
### ⏱️ Checkpoint 2 of 3 — Think · Pair · Explain

Predict what happens when the input size doubles. Justify the trend with an operation count or complexity class—not timing alone.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_2_response = ""  # write at least 8 words
studio_pulse(2, checkpoint_2_response)

---
## Part 9: Common Errors

Let’s look at common mistakes when working with graphs.

**Figure 9.1** — Common error: forgetting to track visited nodes (infinite loop!)

In [ ]:
# ❌ BFS without visited set — INFINITE LOOP on graphs with cycles!
# (We add a safety limit here so it doesn't run forever)

from collections import deque

def bfs_broken(graph, start, max_steps=20):
    """BFS WITHOUT visited tracking — will loop forever!"""
    queue = deque([start])
    order = []
    steps = 0
    
    while queue and steps < max_steps:
        current = queue.popleft()
        order.append(current)
        steps += 1
        
        # Adding ALL neighbors without checking if visited
        for neighbor in graph[current]:
            queue.append(neighbor)  # ❌ No visited check!
    
    return order


small_graph = {"A": ["B"], "B": ["A"]}  # Simple cycle: A ↔ B

print("❌ BFS without visited set (stopped after 20 steps):")
result = bfs_broken(small_graph, "A")
print(f"   {result}")
print("   → It keeps bouncing between A and B forever!")
print()
print("✅ Always use a 'visited' set to avoid revisiting nodes.")

**Figure 9.2** — Common error: KeyError when node is not in the graph

In [ ]:
# ❌ Accessing a node that doesn't exist
graph = {"A": ["B"], "B": ["A"]}

try:
    neighbors = graph["C"]  # C is not in the graph!
except KeyError as e:
    print(f"❌ KeyError: {e}")
    print("   Node 'C' doesn't exist in the graph!")

print()

# ✅ Safe way: use .get() with a default
neighbors = graph.get("C", [])
print(f"✅ Safe access: graph.get('C', []) = {neighbors}")

# ✅ Or check first
if "C" in graph:
    print(graph["C"])
else:
    print("✅ 'C' is not in the graph")

**Figure 9.3** — Common error: undirected graph with only one direction

In [ ]:
# ❌ Forgetting to add the reverse edge in an undirected graph
graph_broken = {
    "A": ["B"],  # A → B
    "B": []      # ❌ Missing B → A!
}

print("Broken undirected graph:")
print(f"  A's neighbors: {graph_broken['A']}")
print(f"  B's neighbors: {graph_broken['B']} ← B can't reach A!")

print()

# ✅ Correct: add edges in both directions
graph_fixed = {
    "A": ["B"],
    "B": ["A"]
}
print("Fixed undirected graph:")
print(f"  A's neighbors: {graph_fixed['A']}")
print(f"  B's neighbors: {graph_fixed['B']}")
print()
print("✅ Tip: Use an add_edge() helper that adds both directions.")

---
## 🌉 Bridge to Next Week

This week we learned how to represent and traverse graphs using BFS and DFS. BFS can find the **shortest path by number of edges**.

But what if edges have different **weights** (distances, costs, time)?

**Next week (Week 12):** We’ll learn **Dijkstra’s Algorithm** to find the shortest path in **weighted graphs**.

| This Week | Next Week |
|-----------|----------|
| Unweighted edges | Weighted edges (distances, costs) |
| BFS finds shortest path (fewest steps) | Dijkstra finds shortest path (lowest total weight) |
| Uses a simple queue | Uses a priority queue (heapq!) |

---
## Part 10: Performance Benchmark — BFS vs DFS

> **🎯 Predict first, then measure. Does reality match your prediction?**
>
> Both BFS and DFS visit every node and edge once, giving O(V + E). Before running, predict: will they take exactly the same time? Will one be consistently faster? Why might there be a difference despite the same big-O?

How do BFS and DFS compare in speed on graphs of increasing size? Let's generate random graphs with approximately 3 edges per node and measure traversal time for both algorithms.

In [ ]:
import time
import random
import matplotlib.pyplot as plt
from collections import deque

def generate_random_graph(n, avg_edges=3):
    """Generate a random undirected graph with n nodes and ~avg_edges edges per node."""
    graph = {i: [] for i in range(n)}
    total_edges = (n * avg_edges) // 2
    for _ in range(total_edges):
        u = random.randint(0, n - 1)
        v = random.randint(0, n - 1)
        if u != v:
            graph[u].append(v)
            graph[v].append(u)
    return graph

sizes = [100, 500, 1000, 2000, 5000]
num_runs = 5

bfs_times = []
dfs_times = []

for n in sizes:
    # BFS benchmark
    times = []
    for _ in range(num_runs):
        g = generate_random_graph(n)
        start = time.time()
        bfs_quiet(g, 0)
        times.append(time.time() - start)
    bfs_times.append(sum(times) / num_runs)

    # DFS benchmark
    times = []
    for _ in range(num_runs):
        g = generate_random_graph(n)
        start = time.time()
        dfs_quiet(g, 0)
        times.append(time.time() - start)
    dfs_times.append(sum(times) / num_runs)

    print(f"n={n:5d}: BFS={bfs_times[-1]:.5f}s  DFS={dfs_times[-1]:.5f}s")

plt.figure(figsize=(10, 6))
plt.plot(sizes, bfs_times, 'o-', label='BFS', linewidth=2)
plt.plot(sizes, dfs_times, 's-', label='DFS', linewidth=2)
plt.xlabel('Number of Nodes (n)')
plt.ylabel('Average Time (seconds)')
plt.title('BFS vs DFS Traversal Time on Random Graphs (~3 edges/node)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

**Observations:**
- Both BFS and DFS have the same theoretical time complexity of **O(V + E)**, and the benchmark confirms they scale similarly as the graph grows.
- BFS uses a `deque` for efficient queue operations, while DFS uses a list as a stack. In practice, BFS may be slightly faster due to `deque.popleft()` being O(1), whereas DFS's `stack.pop()` is also O(1).
- The near-linear growth in time confirms the O(V + E) complexity. Since each node has roughly 3 edges, E is about 3V, so the total work is proportional to n.

---
## 🎢 Exercises

### Easy Exercises

**EX1 (Easy):** Create a graph (adjacency list) for the following connections and print each node's neighbors:
- A connects to B, C
- B connects to A, D
- C connects to A, D, E
- D connects to B, C
- E connects to C

Expected Output:
```
A: ['B', 'C']
B: ['A', 'D']
C: ['A', 'D', 'E']
D: ['B', 'C']
E: ['C']
```

<details><summary>💡 Hint</summary>
Create a dictionary where each key is a node and each value is a list of neighbors.
</details>

In [ ]:
# ✏️ [EX1] Your code here


**EX2 (Easy):** Run BFS on the graph from EX1 starting from node "A". Print the order of visited nodes.

Expected Output:
```
BFS from A: ['A', 'B', 'C', 'D', 'E']
```

<details><summary>💡 Hint</summary>
Use the bfs_quiet function from Part 6 or write your own BFS with a deque and visited set.
</details>

In [ ]:
# ✏️ [EX2] Your code here


**EX3 (Easy):** Run DFS on the same graph from EX1 starting from node "A". Print the order of visited nodes.

Expected Output (may vary depending on neighbor order):
```
DFS from A: ['A', 'C', 'E', 'D', 'B']
```

<details><summary>💡 Hint</summary>
Use the dfs_quiet function from Part 6 or write your own DFS with a stack (list) and visited set.
</details>

In [ ]:
# ✏️ [EX3] Your code here


**EX4 (Easy):** Write a function `count_edges(graph)` that counts the total number of edges in an undirected graph. Test it with your graph from EX1.

Expected Output:
```
Total edges: 5
```

<details><summary>💡 Hint</summary>
Sum up the lengths of all neighbor lists, then divide by 2 (since each edge is counted twice in an undirected graph).
</details>

In [ ]:
# ✏️ [EX4] Your code here


### Medium Exercises

**EX5 (Medium):** Write a function `has_path(graph, start, end)` that returns True if there is a path between start and end, and False otherwise. Use BFS or DFS.

Test with a graph that has **two disconnected components**:
- Component 1: A-B-C
- Component 2: D-E

Expected Output:
```
Path A → C: True
Path A → D: False
```

<details><summary>💡 Hint</summary>
Use BFS/DFS to find all reachable nodes from start. If end is among them, return True.
</details>

In [ ]:
# ✏️ [EX5] Your code here


**EX6 (Medium):** Find the shortest path between two nodes using BFS. Build this graph:

```
1 — 2 — 5
|       |
3 — 4 — 6
```

Find the shortest path from 1 to 6.

Expected Output:
```
Shortest path 1 → 6: [1, 3, 4, 6]
Length: 3
```

<details><summary>💡 Hint</summary>
Use the bfs_shortest_path function from Part 7. Store (node, path) pairs in the queue.
</details>

In [ ]:
# ✏️ [EX6] Your code here


---
### ⏱️ Checkpoint 3 of 3 — Think · Pair · Explain

Choose one completed core exercise. Explain why the algorithm is correct, its dominant cost, and one mechatronics situation where that cost matters.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_3_response = ""  # write at least 8 words
studio_pulse(3, checkpoint_3_response)

---
## 🌟 Optional Extension

Exercises 7 and above are enrichment for remaining class time or independent curiosity. They are not homework and are not collected.


**EX7 (Medium):** Write a function `is_connected(graph)` that returns True if the graph is fully connected (every node can reach every other node).

Test with:
- Graph 1: `{"A": ["B"], "B": ["A", "C"], "C": ["B"]}` → True
- Graph 2: `{"A": ["B"], "B": ["A"], "C": ["D"], "D": ["C"]}` → False

Expected Output:
```
Graph 1 connected: True
Graph 2 connected: False
```

<details><summary>💡 Hint</summary>
Run BFS/DFS from any node. If the number of visited nodes equals the total number of nodes, the graph is connected.
</details>

In [ ]:
# ✏️ [EX7] Your code here


**EX8 (Medium):** Write a function `find_all_paths(graph, start, end)` that returns ALL possible paths (not just the shortest) between two nodes.

Test with a small graph: A-B, A-C, B-D, C-D. Find all paths from A to D.

Expected Output:
```
All paths A → D:
  A → B → D
  A → C → D
```

<details><summary>💡 Hint</summary>
Use DFS with backtracking. Keep a path list and a visited set. When you reach the end, save the path. After exploring a neighbor, remove it from visited (backtrack).
</details>

In [ ]:
# ✏️ [EX8] Your code here


**EX9 (Medium):** Write a function `count_components(graph)` that counts how many disconnected components a graph has.

Test with: `{"A": ["B"], "B": ["A"], "C": ["D"], "D": ["C"], "E": []}`

Expected Output:
```
Number of components: 3
```

<details><summary>💡 Hint</summary>
Keep a global visited set. For each unvisited node, run BFS/DFS (that's one component). Count how many times you start a new BFS/DFS.
</details>

In [ ]:
# ✏️ [EX9] Your code here


**EX10 (Medium):** Create a directed graph for course prerequisites:
- "Math" must be taken before "Physics"
- "Math" must be taken before "CS101"
- "CS101" must be taken before "CS201"
- "Physics" must be taken before "CS201"

Use DFS to find all courses that must be taken before "CS201" (all ancestors).

Expected Output:
```
Prerequisites for CS201: ['Math', 'Physics', 'CS101']
```

<details><summary>💡 Hint</summary>
Build a reverse graph (if A→B exists, add B→A). Then run DFS from CS201 on the reverse graph to find all prerequisites.
</details>

In [ ]:
# ✏️ [EX10] Your code here


### Challenge Exercises

**EX11 (Challenge):** Write a function `has_cycle(graph)` that detects if an undirected graph has a cycle. Return True if a cycle exists.

Test with:
- `{"A": ["B"], "B": ["A", "C"], "C": ["B"]}` → False (no cycle, it's a line)
- `{"A": ["B", "C"], "B": ["A", "C"], "C": ["A", "B"]}` → True (A-B-C-A is a cycle)

Expected Output:
```
Graph 1 has cycle: False
Graph 2 has cycle: True
```

<details><summary>💡 Hint</summary>
Use BFS/DFS. Track visited nodes AND the parent of each node. If you encounter a visited neighbor that is NOT the parent, you've found a cycle.
</details>

In [ ]:
# ✏️ [EX11] Your code here


**EX12 (Challenge):** Implement a simple **maze solver** using BFS. The maze is a 2D grid where `0` is a path and `1` is a wall. Find the shortest path from top-left `(0,0)` to bottom-right `(rows-1, cols-1)`.

```python
maze = [
    [0, 0, 1, 0],
    [1, 0, 1, 0],
    [0, 0, 0, 0],
    [0, 1, 1, 0]
]
```

Expected Output:
```
Shortest path: [(0,0), (0,1), (1,1), (2,1), (2,2), (2,3), (3,3)]
Steps: 6
```

<details><summary>💡 Hint</summary>
Treat each cell as a node. Neighbors are the 4 adjacent cells (up, down, left, right) that are within bounds and have value 0. Use BFS with (row, col) tuples.
</details>

In [ ]:
# ✏️ [EX12] Your code here
